# 1. Carregamento da base
Leitura do arquivo Excel e visualização inicial do DataFrame.

In [10]:
import pandas as pd

arquivo = "MLCQCodeSmellSamples.xlsx"
df = pd.read_excel(arquivo)

print(df.shape)
df.head()

(14739, 15)


,id,reviewer_id,sample_id,smell,severity,review_timestamp,type,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project
0,526,6,5771277,feature envy,none,2019-03-27 10:34:53.041496,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1.0
1,527,6,5771277,long method,none,2019-03-27 10:34:53.042443,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1.0
2,528,6,5786929,blob,critical,2019-03-27 10:37:38.107923,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1.0
3,529,6,5786929,data class,critical,2019-03-27 10:37:38.109068,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1.0
4,530,6,5788107,feature envy,none,2019-03-27 10:37:49.627100,function,org.apache.tika.parser.ocr.TesseractOCRConfig#...,git@github.com:apache/tika.git,4131c6e30f2e0eb1feb85e0f7576531d4e830468,/tika-parsers/src/main/java/org/apache/tika/pa...,531,534,https://github.com/apache/tika/blob/4131c6e30f...,1.0


## 1.1 Conferência das colunas
Lista os nomes das colunas para validar a estrutura da base.

In [11]:
df.columns.tolist()

['id',
 'reviewer_id',
 'sample_id',
 'smell',
 'severity',
 'review_timestamp',
 'type',
 'code_name',
 'repository',
 'commit_hash',
 'path',
 'start_line',
 'end_line',
 'link',
 'is_from_industry_relevant_project']

## 1.2 Tamanho antes da limpeza
Exibe a quantidade de linhas e colunas antes da deduplicação.

In [12]:
print(df.shape)

(14739, 15)


## 1.3 Remoção de duplicatas
Remove registros com link repetido para evitar verificações redundantes.

In [13]:
df = df.drop_duplicates(subset=["link"])
print(df.shape)

(4770, 15)


# 2. Validação de links HTTP
Importa bibliotecas para requisições e processamento paralelo.

In [14]:
import requests
from requests.exceptions import RequestException
from concurrent.futures import ThreadPoolExecutor, as_completed

## 2.1 Função de verificação individual
Define uma rotina para consultar URL e retornar o status HTTP.

In [15]:
def verificar_link(url, timeout=8):
    try:
        response = requests.get(url, timeout=timeout, allow_redirects=True)
        return response.status_code
    except RequestException:
        return None

## 2.2 Função de verificação em lote
Processa várias URLs em paralelo e registra o progresso da execução.

In [16]:
def verificar_links_em_lote(urls, max_workers=10):
    resultados = {}
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {executor.submit(verificar_link, url): url for url in urls}
        
        total = len(futuros)
        concluidos = 0
        
        for futuro in as_completed(futuros):
            url = futuros[futuro]
            try:
                resultados[url] = futuro.result()
            except Exception:
                resultados[url] = None
            
            concluidos += 1
            if concluidos % 100 == 0 or concluidos == total:
                print(f"{concluidos}/{total} links verificados")
    
    return resultados

## 2.3 Padronização dos links
Normaliza o campo de link e gera uma lista única para validação.

In [17]:
df.columns = df.columns.str.strip()

LINK_COL = "link"

df[LINK_COL] = df[LINK_COL].astype(str).str.strip()
df = df[df[LINK_COL].notna()]
df = df[df[LINK_COL] != ""]
df = df[df[LINK_COL].str.lower() != "nan"]

urls_unicas = df[LINK_COL].unique().tolist()
print(f"Total de links únicos: {len(urls_unicas)}")

Total de links únicos: 4770


## 2.4 Execução da verificação
Executa a checagem das URLs e salva os resultados por link.

In [18]:
resultados = verificar_links_em_lote(urls_unicas, max_workers=10)

100/4770 links verificados
200/4770 links verificados
300/4770 links verificados
400/4770 links verificados
500/4770 links verificados
600/4770 links verificados
700/4770 links verificados
800/4770 links verificados
900/4770 links verificados
1000/4770 links verificados
1100/4770 links verificados
1200/4770 links verificados
1300/4770 links verificados
1400/4770 links verificados
1500/4770 links verificados
1600/4770 links verificados
1700/4770 links verificados
1800/4770 links verificados
1900/4770 links verificados
2000/4770 links verificados
2100/4770 links verificados
2200/4770 links verificados
2300/4770 links verificados
2400/4770 links verificados
2500/4770 links verificados
2600/4770 links verificados
2700/4770 links verificados
2800/4770 links verificados
2900/4770 links verificados
3000/4770 links verificados
3100/4770 links verificados
3200/4770 links verificados
3300/4770 links verificados
3400/4770 links verificados
3500/4770 links verificados
3600/4770 links verificados
3

## 2.5 Mapeamento de status
Associa o status HTTP no DataFrame e cria o indicador de link válido.

In [19]:
df["status_code"] = df[LINK_COL].map(resultados)
df["link_ok"] = df["status_code"] == 200

## 2.6 Diagnóstico inicial
Mostra a distribuição dos códigos HTTP obtidos na primeira rodada.

In [20]:
df["status_code"].value_counts(dropna=False)

status_code
200.0    2921
429.0    1582
404.0     264
NaN         2
502.0       1
Name: count, dtype: int64

## 2.7 Coleta de links 429
Separa as URLs com status 429 para reprocessamento controlado.

In [21]:
urls_nao_200 = df.loc[df["status_code"] != 200, LINK_COL].dropna().unique().tolist()
print(f"Links com status diferente de 200: {len(urls_nao_200)}")

Links com status diferente de 200: 1849


## 2.8 Função de retry
Implementa tentativas com espera progressiva para reduzir falhas temporárias.

In [22]:
import time
import requests
from requests.exceptions import RequestException

def verificar_link_com_retry(url, timeout=8, tentativas=3, espera=3):
    for tentativa in range(1, tentativas + 1):
        try:
            response = requests.get(url, timeout=timeout, allow_redirects=True)
            status = response.status_code

            if status == 200:
                return 200

            if tentativa < tentativas:
                time.sleep(espera * tentativa)

        except RequestException:
            if tentativa < tentativas:
                time.sleep(espera * tentativa)
            else:
                return None

    return status

## 2.9 Reprocessamento em lote
Prepara uma rotina paralela para revalidar links que retornaram 429.

In [23]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def verificar_links_reprocessamento(urls, max_workers=1):
    resultados = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {executor.submit(verificar_link_com_retry, url): url for url in urls}

        total = len(futuros)
        concluidos = 0

        for futuro in as_completed(futuros):
            url = futuros[futuro]
            try:
                resultados[url] = futuro.result()
            except Exception:
                resultados[url] = None

            concluidos += 1
            if concluidos % 50 == 0 or concluidos == total:
                print(f"{concluidos}/{total} links não-200 reprocessados")

    return resultados

## 2.10 Execução do retry
Executa a segunda rodada de verificação somente para links com status 429.

In [24]:
resultados_retry = verificar_links_reprocessamento(urls_nao_200, max_workers=1)

50/1849 links não-200 reprocessados
100/1849 links não-200 reprocessados
150/1849 links não-200 reprocessados
200/1849 links não-200 reprocessados
250/1849 links não-200 reprocessados
300/1849 links não-200 reprocessados
350/1849 links não-200 reprocessados
400/1849 links não-200 reprocessados
450/1849 links não-200 reprocessados
500/1849 links não-200 reprocessados
550/1849 links não-200 reprocessados
600/1849 links não-200 reprocessados
650/1849 links não-200 reprocessados
700/1849 links não-200 reprocessados
750/1849 links não-200 reprocessados
800/1849 links não-200 reprocessados
850/1849 links não-200 reprocessados
900/1849 links não-200 reprocessados
950/1849 links não-200 reprocessados
1000/1849 links não-200 reprocessados
1050/1849 links não-200 reprocessados
1100/1849 links não-200 reprocessados
1150/1849 links não-200 reprocessados
1200/1849 links não-200 reprocessados
1250/1849 links não-200 reprocessados
1300/1849 links não-200 reprocessados
1350/1849 links não-200 reproces

## 2.11 Atualização dos resultados
Aplica os novos status no DataFrame e recalcula a coluna link_ok.

In [25]:
for url, novo_status in resultados_retry.items():
    df.loc[df[LINK_COL] == url, "status_code"] = novo_status

df["link_ok"] = df["status_code"] == 200

## 2.12 Diagnóstico final
Reconta os códigos HTTP após o reprocessamento dos links com status 429.

In [26]:
df["status_code"].value_counts(dropna=False)

status_code
200.0    4364
404.0     406
Name: count, dtype: int64

# 3. Geração dos artefatos finais
Filtra os registros válidos e exporta os arquivos de saída.

In [27]:
df_limpo = df[df["status_code"] == 200].copy()
print(df_limpo.shape)

(4364, 17)


## 3.1 Exportação dos arquivos
Salva a base completa com status e a base limpa com status 200.

In [28]:
df.to_excel("MLCQ_status.xlsx", index=False)
df_limpo.to_excel("MLCQ_status_200.xlsx", index=False)